In [1]:
import os
import re
import numpy as np
import tifffile
from pathlib import Path
import pandas as pd
from tifffile import imwrite
from scipy.spatial import KDTree

def crop_tiff_stack(input_dir, output_dir, projXY_path, projXZ_path, crop=True):
    try:
        # Load projections
        projXY = tifffile.imread(projXY_path)
        projXZ = tifffile.imread(projXZ_path)

        # Ensure output directory exists
        os.makedirs(output_dir, exist_ok=True)

        # Regex to find frame info
        pattern = re.compile(r'frame-(\d+)\.tif')

        # Find available frames
        frame_numbers = set()
        for file in os.listdir(input_dir):
            match = pattern.search(file)
            if match:
                frame_numbers.add(int(match.group(1)))

        if not frame_numbers:
            raise ValueError("No frames found in the input directory.")

        frames = sorted(frame_numbers)

        for t in frames:
            ref_path = os.path.join(input_dir, f"Channel-ref-frame-{t}.tif")
            crc_path = os.path.join(input_dir, f"Channel-CRC-frame-{t}.tif")
            wt_path = os.path.join(input_dir, f"Channel-WT-frame-{t}.tif")
            
            if os.path.exists(ref_path):
                print(f"Processing frame {t}")

                if crop:
                    frame = tifffile.imread(ref_path)
                    print(f"Original shape: {frame.shape}")

                    # Determine the number of channels
                    if frame.ndim == 4:
                        num_channels = frame.shape[1]
                    else:
                        num_channels = 1
                        frame = frame[:, np.newaxis, :, :]

                    # Initialize cropping limits
                    slice_min, slice_max = 0, frame.shape[0]
                    row_min, row_max = 0, frame.shape[2]
                    col_min, col_max = 0, frame.shape[3]

                    # Create masks based on projections
                    maskXY = projXY[t] > 0
                    maskXZ = projXZ[t] > 0

                    # Find coordinates of non-zero pixels
                    coordsXY = np.where(maskXY)
                    coordsXZ = np.where(maskXZ)

                    # Adjust cropping limits based on XY projection
                    if coordsXY[0].size > 0:
                        row_min, row_max = np.min(coordsXY[0]), np.max(coordsXY[0]) + 1
                        col_min, col_max = np.min(coordsXY[1]), np.max(coordsXY[1]) + 1

                    # Adjust cropping limits based on XZ projection
                    if coordsXZ[0].size > 0:
                        slice_min, slice_max = np.min(coordsXZ[0]), np.max(coordsXZ[0]) + 1

                    # Process each channel
                    for channel_path, channel_name in zip([ref_path, crc_path, wt_path], ["ref", "CRC", "WT"]):
                        if os.path.exists(channel_path):
                            frame = tifffile.imread(channel_path)
                            if frame.ndim == 3:
                                frame = frame[:, np.newaxis, :, :]
                            
                            cropped = frame[slice_min:slice_max, :, row_min:row_max, col_min:col_max]
                            print(f"Cropped shape ({channel_name}): {cropped.shape}")

                            # Ensure the correct shape: (T, Z, C, Y, X, S)
                            cropped = np.expand_dims(cropped, axis=0)  # Add T dimension
                            cropped = np.expand_dims(cropped, axis=-1) # Add S dimension
                            
                            output_path = os.path.join(output_dir, f"Channel-{channel_name}-frame-{t}.tif")
                            tifffile.imwrite(
                                output_path,
                                cropped,
                                imagej=True,
                                metadata={'axes': 'TZCYXS'},
                            )
                        else:
                            print(f"Warning: {channel_path} not found.")
            else:
                print(f"Frame {t} not found: {ref_path}")

        print(f"Frames cropped and saved to {output_dir}")

    except FileNotFoundError as e:
        print(f"Error: {e}")
    except ValueError as e:
        print(f"Error: {e}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

def compute_knn_features(points, k=5):
    if len(points) == 0:
        return [], [], []

    setX = points['cx'] / points['cx'].max()
    setY = points['cy'] / points['cy'].max()
    setZ = points['cz'] / points['cz'].max()

    coords = np.stack([setX, setY, setZ], axis=1)

    tree = KDTree(coords)
    k_use = min(k, len(coords))
    distances, indices = tree.query(coords, k=k_use)

    distAv = []
    pWT = []
    pC = []

    for i in range(len(indices)):
        neighbor_indices = indices[i]
        neighbor_phenotypes = points.iloc[neighbor_indices]['phenotype']

        count_WT = sum(p.startswith('WT') for p in neighbor_phenotypes)
        count_C = sum(p.startswith('C') for p in neighbor_phenotypes)
        total = len(neighbor_phenotypes)

        distAv.append(np.median(distances[i]))
        pWT.append(count_WT)
        pC.append(count_C)

    return distAv, pWT, pC

def stitch3D(mask_list):
    return np.stack(mask_list, axis=0).astype(np.uint16)

In [ ]:
import sys
#
path_codes = 'Z:/users/6331823/Mario Pipeline/codes'
if not path_codes:
    raise ValueError("Tracker4Utrecht not found")

sys.path.append(path_codes)
# Now you can import the functions file
import functions_Tracker4Utrecht_drafting as t4u

In [ ]:
# Main workflow
try:
    pathI, files = t4u.process_directory()
    # metadata = t4u.process_metadata(pathI)
    user_data = t4u.create_widgets(files, pathI)  
    print(user_data)
    #       
except Exception as e:
    print(f"Error: {e}")
    

In [ ]:
# Example Usage (replace with your actual paths):
input_directory = pathI
output_directory = os.path.join(str(input_directory), user_data['output_name'])
projXY_file = user_data['projXY_path']
projXZ_file = user_data['projXZ_path']

In [ ]:
crop_tiff_stack(input_directory, output_directory, projXY_file, projXZ_file)

Processing frame 0
Original shape: (81, 996, 1020)
Cropped shape (ref): (11, 1, 189, 186)
Cropped shape (CRC): (11, 1, 189, 186)
Cropped shape (WT): (11, 1, 189, 186)
Processing frame 1
Original shape: (81, 996, 1020)
Cropped shape (ref): (11, 1, 194, 186)
Cropped shape (CRC): (11, 1, 194, 186)
Cropped shape (WT): (11, 1, 194, 186)
Processing frame 2
Original shape: (81, 996, 1020)
Cropped shape (ref): (13, 1, 194, 186)
Cropped shape (CRC): (13, 1, 194, 186)
Cropped shape (WT): (13, 1, 194, 186)
Processing frame 3
Original shape: (81, 996, 1020)
Cropped shape (ref): (13, 1, 194, 186)
Cropped shape (CRC): (13, 1, 194, 186)
Cropped shape (WT): (13, 1, 194, 186)
Processing frame 4
Original shape: (81, 996, 1020)
Cropped shape (ref): (13, 1, 244, 200)
Cropped shape (CRC): (13, 1, 244, 200)
Cropped shape (WT): (13, 1, 244, 200)
Processing frame 5
Original shape: (81, 996, 1020)
Cropped shape (ref): (13, 1, 235, 204)
Cropped shape (CRC): (13, 1, 235, 204)
Cropped shape (WT): (13, 1, 235, 204

with the codes of above, you have a crop folder for each TL file, with the code of below you can run the 2D model and after of this OCAR/stitch from cellpose to get the 3D files

In [ ]:
import os
import numpy as np
import pandas as pd
import re
import tifffile
import warnings
import cellpose
from cellpose import models
import torch
from scipy.ndimage import zoom
from skimage.measure import regionprops, regionprops_table
import esda
import libpysal

rescale_factor = 0.5  ## about speed optimization
customModel = False

# define your directory 
input_directory = r"Z:\users\6331823\Mario Pipeline\frames-frames_extracted\Second try"
output_directory = os.path.join(input_directory, "cellpose")

# Verifica que la carpeta de salida exista
if not os.path.exists(output_directory):
    os.makedirs(output_directory)

# Configuración del modelo
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device.upper()} for processing.")

# Ruta al modelo personalizado
if customModel == True:
    custom_model_path = r"Z:\users\6331823\Mario Pipeline\codes\models\organoids_3D.pkl"

    if os.path.exists(custom_model_path):
        model = models.CellposeModel(gpu=(device == "cuda"), pretrained_model=custom_model_path)
        print(f"Using custom model: {custom_model_path}")
    else:
        warnings.warn("Custom model not found. Using the default nuclei model.")
        model = models.Cellpose(gpu=(device == "cuda"), model_type="nuclei")
else:
    model = models.CellposeModel(gpu=(device == "cuda"))
    print("Using SAM standard model")

# Lista de archivos en el directorio
files = sorted([f for f in os.listdir(input_directory) if f.endswith(".tif") and f.startswith("Channel-ref-")])
# files = sorted([f for f in os.listdir(input_directory) if f.endswith(".tif")])

summary_results = []

for file in files:
# for file in files[2:]:  # Empieza desde el tercer archivo (índice 2)
    frame_path = os.path.join(input_directory, file)
    print(f"Processing {file}...")

    frame = tifffile.imread(frame_path)
    #print(f"Original shape: {frame.shape}")

    # Extraer el canal 1 si es una imagen con múltiples canales
    if frame.ndim == 4:  # Multi-channel case (Z, C, Y, X)
        channel1_frame = frame[:, 0, :, :]  # Extrae solo el canal 1
    else:
        channel1_frame = frame  # Si es monocanal, usa directamente
    #print(f"Channel 1 shape: {channel1_frame.shape}")
    
    depthIm, heightIm, widthIm = channel1_frame.shape
    original_shape =  (depthIm, heightIm, widthIm)
    
     # Reescalar imagen
    channel1_frame = zoom(channel1_frame, zoom=rescale_factor, order=1) # Escalar en las tres dimensiones
    
    depthIm, heightIm, widthIm = channel1_frame.shape
    
    print(f"Original shape: {frame.shape}, Resized shape: {channel1_frame.shape}")
          
    # Calcular el umbral
    mean_intensity = np.mean(channel1_frame)
    std_intensity = np.std(channel1_frame)
    threshold = mean_intensity + std_intensity

    # Aplicar el umbral
    channel1_frame[channel1_frame < threshold] = 0
    
    shapeW = zoom(channel1_frame, zoom = (1,0.5,0.5), order=1)
    w = libpysal.weights.lat2W(shapeW.shape[1], shapeW.shape[2])
    masks_list = []
    
    for z in range(depthIm):
        #print(f"slice.. {z}")
        plane_2d = channel1_frame[z, :, :]
        planeMoran = shapeW[z, :, :]
        moran = esda.Moran(planeMoran.ravel(), w)
        if moran.I > 0.6:
            print(f"Slice {z}, moran: {moran.I}")
            masks, _, _ = model.eval(plane_2d, diameter=None, do_3D=False)
            # full_mask_3d[z, :, :] = masks.astype(np.uint16)
            
        else:
            # full_mask_3d[z, :, :] = 0
            masks = np.zeros_like(plane_2d, dtype=np.uint16)
            print(f"Skipping slice {z} (low signal), moran: {moran.I}")
            
        masks_list.append(masks)
    #

    full_mask_3d = np.stack(masks_list, axis=0).astype(np.uint16)
    full_mask_3d = cellpose.utils.stitch3D(full_mask_3d)
    
    # Volver al tamaño original
    zoom_factors = np.array(original_shape) / np.array(full_mask_3d.shape)
    mask_original = zoom(full_mask_3d, zoom=zoom_factors, order=0).astype(np.uint16)    
    #    
    # Guardar la imagen segmentada
    segmented_tiff_file = os.path.join(output_directory, f"{file.split('.')[0]}_segmented.tif")
    tifffile.imwrite(segmented_tiff_file, mask_original)
    print(f"Segmented masks saved: {segmented_tiff_file}")
    #
    # Extraer propiedades
    props = regionprops(mask_original)
    data = []
    for prop in props:
        label = prop.label
        centroid = prop.centroid  # (z, y, x)
        bounding_box = prop.bbox  # (min_z, min_y, min_x, max_z, max_y, max_x)
        volume = prop.area

        data.append({
            'Label': label,
            'z': centroid[0],
            'y': centroid[1],
            'x': centroid[2],
            'BoundingBox': bounding_box,
            'Volume': volume
        })

    # Guardar como .txt separado por tabs
    df_props = pd.DataFrame(data).rename(columns={'Label': 'label'})
    output_txt_file = os.path.join(output_directory, f"{file.split('.')[0]}_props.txt")
    df_props.to_csv(output_txt_file, sep='\t', index=False)
    print(f"Properties saved to: {output_txt_file}")   

    # Canales CRC y WT

    frameNumber = re.search(r'frame-(\d+)', file).group(1)
    ch_wt = tifffile.imread(os.path.join(input_directory, f"Channel-WT-frame-{frameNumber}.tif"))
    ch_crc = tifffile.imread(os.path.join(input_directory, f"Channel-CRC-frame-{frameNumber}.tif"))

    props_wt = regionprops_table(mask_original, intensity_image=ch_wt,
                                 properties=['label', 'area', 'mean_intensity'])
    props_crc = regionprops_table(mask_original, intensity_image=ch_crc,
                                  properties=['label', 'area', 'mean_intensity'])

    df_wt = pd.DataFrame(props_wt).rename(columns={'area': 'Area_WT', 'mean_intensity': 'Raw_WT'})
    df_crc = pd.DataFrame(props_crc).rename(columns={'area': 'Area_CRC', 'mean_intensity': 'Raw_CRC'})

    # Combinar todo
    df_final = df_props.merge(df_wt, on='label').merge(df_crc, on='label')
    
    # Assign phentoypes according to the defined thresholds
    phenotype = []
    count_phenotype_c1 = 0
    count_phenotype_c2 = 0
    count_phenotype_c1and2 = 0
    count_phenotype_empty = 0
    k_channel1 = 0
    k_channel2 = 0.5 

    for _, row in df_final.iterrows():
        VolTot = row['Volume']
        VolCh1 = row['Area_WT']
        avgCh1 = row['Raw_WT'] / VolCh1 if VolCh1 > 0 else 0
        VolCh2 = row['Area_CRC']
        avgCh2 = row['Raw_CRC'] / VolCh2 if VolCh2 > 0 else 0

        pheno = "-"
        qa = 0
        qb = 0

        if VolCh1 > k_channel1 * VolTot:
            # Fenotipo WT
            pheno_c1 = "WT"
            count_phenotype_c1 += 1
            qa = 1
        else:
            pheno_c1 = "-"

        if VolCh2 > k_channel2 * VolTot:
            # Fenotipo CRC
            pheno_c2 = "CRC"
            pheno = "CRC"
            count_phenotype_c2 += 1
            qb = 1
        else:
            pheno_c2 = "-"

        if qa + qb == 2:
            if avgCh1 >= 2 * avgCh2:
                pheno = "WT"
            else:
                pheno = "CRC"
            count_phenotype_c1and2 += 1
        elif pheno == "-":
            count_phenotype_empty += 1

        if qa == 1 and qb == 0:
            pheno = "WT"

        phenotype.append(pheno)

    df_final['Phenotype'] = phenotype
    summary_results.append({
        'file': file,
        'phenotype_count_c1': count_phenotype_c1,
        'phenotype_count_c2': count_phenotype_c2,
        'phenotype_count_c1and2': count_phenotype_c1and2,
        'phenotype_count_empty': count_phenotype_empty
    })  

    # Guardar CSV
    output_csv = os.path.join(output_directory, f"{file.split('.')[0]}_quantified.csv")
    df_final.to_csv(output_csv, index=False)
    print(f"Quantified data saved to: {output_csv}")

summary_results = pd.DataFrame(summary_results)
summary_csv = os.path.join(output_directory, "summary_results.csv")
summary_results.to_csv(summary_csv, index=False)

Stitching the slices using cellpose stitch3D

In [ ]:
import os
import tifffile
import cellpose

input_directory = r"Z:\users\6331823\Mario Pipeline\frames-frames_extracted\Second try\cellpose"
output_directory = os.path.join(input_directory, "stitched")

# Make output dir if non-existent
if not os.path.exists(output_directory):
    os.makedirs(output_directory)

# Lista de archivos en el directorio
files = sorted([f for f in os.listdir(input_directory) if f.endswith(".tif")])

for file in files:
    frameNumber = re.search(r'frame-(\d+)', file).group(1)
    print(f"Stitching frame {frameNumber}")

    frame = tifffile.imread(file)
    stitched = cellpose.utils.stitch3D(frame)
    
    savePath = os.path.join(output_directory, f"Channel-ref-frame-{frameNumber}_stitched.tif")
    tifffile.imwrite(savePath, stitched)

and this is just a speucode that introduces the thigns to analyze the channels

In [ ]:
import os
import numpy as np
import tifffile
import warnings
import pandas as pd
import torch
from scipy.ndimage import zoom
from skimage.measure import regionprops, regionprops_table
from cellpose import models

# ----------------- Configuración -----------------

k_channel1 = 0
k_channel2 = 0.5


rescale_factor = 0.5
input_directory = r"E:\organoids_paper\2025\frames-nuclei"
output_directory = os.path.join(input_directory, "cellpose")
custom_model_path = r"E:\organoids_paper\models\organoids_3D.pkl"

os.makedirs(output_directory, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device.upper()} for processing.")

# Cargar modelo Cellpose
if os.path.exists(custom_model_path):
    model = models.CellposeModel(gpu=(device == "cuda"), pretrained_model=custom_model_path)
    print(f"Using custom model: {custom_model_path}")
else:
    warnings.warn("Custom model not found. Using the default nuclei model.")
    model = models.Cellpose(gpu=(device == "cuda"), model_type="nuclei")

# ----------------- Función para reconstruir máscara -----------------

def stitch3D(mask_list):
    return np.stack(mask_list, axis=0).astype(np.uint16)

# ----------------- Procesamiento -----------------

files = sorted([f for f in os.listdir(input_directory) if f.endswith(".tif")])

for file in files:
    print(f"\nProcessing {file}...")
    frame_path = os.path.join(input_directory, file)
    raw_img = tifffile.imread(frame_path)  # (Z, C, Y, X)

    if raw_img.ndim != 4 or raw_img.shape[1] < 3:
        print(f"Skipping {file}: Expected shape (Z, C, Y, X) with at least 3 channels.")
        continue

    # Canal de núcleos
    ch_nuc = raw_img[:, 0, :, :]  # (Z, Y, X)
    original_shape = ch_nuc.shape

    # Reescalar
    ch_nuc_rescaled = zoom(ch_nuc, zoom=rescale_factor, order=1)

    # Umbralado
    mean = np.mean(ch_nuc_rescaled)
    std = np.std(ch_nuc_rescaled)
    threshold = mean + std
    mask_input = ch_nuc_rescaled.copy()
    mask_input[mask_input < threshold] = 0

    thrs_intensity = np.mean(mask_input)
    masks_list = []

    for z in range(mask_input.shape[0]):
        plane = ch_nuc_rescaled[z]
        plane_check = mask_input[z]
        if np.mean(plane_check) > thrs_intensity:
            masks, _, _ = model.eval(plane, diameter=None, channels=[0, 0], do_3D=False)
        else:
            masks = np.zeros_like(plane, dtype=np.uint16)
        masks_list.append(masks)

    full_mask_3d = stitch3D(masks_list)

    # Volver a tamaño original
    zoom_factors = np.array(original_shape) / np.array(full_mask_3d.shape)
    mask_original = zoom(full_mask_3d, zoom=zoom_factors, order=0).astype(np.uint16)

    # Guardar máscara segmentada
    segmented_path = os.path.join(output_directory, f"{file.split('.')[0]}_segmented.tif")
    tifffile.imwrite(segmented_path, mask_original)
    print(f"Segmented saved: {segmented_path}")

    # Extraer propiedades geométricas
    props = regionprops(mask_original)
    data = [{
        'label': p.label,
        'z': p.centroid[0],
        'y': p.centroid[1],
        'x': p.centroid[2],
        'Volume': p.area,
        'BoundingBox': p.bbox
    } for p in props]
    df_props = pd.DataFrame(data)

    # Canales CRC y WT
    ch_wt = raw_img[:, 1, :, :]
    ch_crc = raw_img[:, 2, :, :]

    props_wt = regionprops_table(mask_original, intensity_image=ch_wt,
                                 properties=['label', 'area', 'mean_intensity'])
    props_crc = regionprops_table(mask_original, intensity_image=ch_crc,
                                  properties=['label', 'area', 'mean_intensity'])

    df_wt = pd.DataFrame(props_wt).rename(columns={'area': 'Area_WT', 'mean_intensity': 'Raw_WT'})
    df_crc = pd.DataFrame(props_crc).rename(columns={'area': 'Area_CRC', 'mean_intensity': 'Raw_CRC'})

    # Combinar todo
    df_final = df_props.merge(df_wt, on='label').merge(df_crc, on='label')
    
    # Assign phentoypes according to the defined thresholds
    phenotype = []
    count_phenotype_c1 = 0
    count_phenotype_c2 = 0
    count_phenotype_c1and2 = 0
    count_phenotype_empty = 0

    for _, row in df_final.iterrows():
        VolTot = row['Volume']
        VolCh1 = row['Area_WT']
        avgCh1 = row['Raw_WT'] / VolCh1 if VolCh1 > 0 else 0
        VolCh2 = row['Area_CRC']
        avgCh2 = row['Raw_CRC'] / VolCh2 if VolCh2 > 0 else 0

        pheno = "-"
        qa = 0
        qb = 0

        if VolCh1 > k_channel1 * VolTot:
            # Fenotipo WT
            pheno_c1 = "WT"
            count_phenotype_c1 += 1
            qa = 1
        else:
            pheno_c1 = "-"

        if VolCh2 > k_channel2 * VolTot:
            # Fenotipo CRC
            pheno_c2 = "CRC"
            pheno = "CRC"
            count_phenotype_c2 += 1
            qb = 1
        else:
            pheno_c2 = "-"

        if qa + qb == 2:
            if avgCh1 >= 2 * avgCh2:
                pheno = "WT"
            else:
                pheno = "CRC"
            count_phenotype_c1and2 += 1
        elif pheno == "-":
            count_phenotype_empty += 1

        if qa == 1 and qb == 0:
            pheno = "WT"

        phenotype.append(pheno)

    df_final['Phenotype'] = phenotype

    # Guardar CSV
    output_csv = os.path.join(output_directory, f"{file.split('.')[0]}_quantified.csv")
    df_final.to_csv(output_csv, index=False)
    print(f"Quantified data saved to: {output_csv}")


if the code is stopped, this will allow you to manage folder with more efficiency, let sayhaha

In [ ]:
import os
import shutil

# Define your folders
source_folder = r'E:\organoids_paper\2025\frames-nuclei'       
analyzed_folder = r'E:\organoids_paper\2025\frames-nuclei\cellpose' 
backup_folder = r'E:\organoids_paper\2025\frames-nuclei\done' 

# Ensure the backup folder exists
os.makedirs(backup_folder, exist_ok=True)

# List all files in the source folder
for filename in os.listdir(source_folder):
    source_file_path = os.path.join(source_folder, filename)
    
    # Skip if it's not a file
    if not os.path.isfile(source_file_path):
        continue
        
    name, ext = os.path.splitext(filename)
    analyzed_filename = f"{name}_segmented{ext}"
    analyzed_file_path = os.path.join(analyzed_folder, analyzed_filename)

    # Check if the analyzed version exists
    if os.path.exists(analyzed_file_path):
        # Move the original file to backup
        shutil.move(source_file_path, os.path.join(backup_folder, filename))
        print(f"Moved: {filename}")
    else:
        print(f"Not analyzed: {filename}")
